In [1]:
print('hello')

hello


In [2]:
#import torch

import pandas as pd
import numpy as np
import os

import json
from sklearn.cluster import KMeans
import umap
import matplotlib.pyplot as plt
import numpy as np


/opt/anaconda3/envs/clustering/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load data
embeddings = np.load("embeddings/embeddings.npy")
labels = np.load("embeddings/labels.npy")
label_names = json.load(open("data/labels.json"))

print(f"Embeddings shape: {embeddings.shape}")

# KMeans with k=101 (one cluster per food class)
print("Running KMeans...")
kmeans = KMeans(n_clusters=101, random_state=42, n_init=10, verbose=1)
cluster_ids = kmeans.fit_predict(embeddings)

print(f"Cluster IDs shape: {cluster_ids.shape}")
print(f"Unique clusters: {len(np.unique(cluster_ids))}")

# Save cluster assignments
np.save("cluster_ids.npy", cluster_ids)
print("Saved cluster_ids.npy")

In [ ]:
# Load data
embeddings = np.load("embeddings/embeddings.npy")
labels = np.load("embeddings/labels.npy")
cluster_ids = np.load("cluster_ids.npy")
label_names = json.load(open("data/labels.json"))

print("Running UMAP...")
reducer = umap.UMAP(n_components=2, random_state=42, verbose=True)
embedding_2d = reducer.fit_transform(embeddings)
print("UMAP done")
np.save("embedding_2d.npy", embedding_2d)
print("Saved embedding_2d.npy")

In [ ]:

embedding_2d = np.load("embedding_2d.npy")
# Plot 1 - colored by true label
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

scatter1 = axes[0].scatter(embedding_2d[:, 0], embedding_2d[:, 1],
                            c=labels, cmap="tab20", s=1, alpha=0.5)
axes[0].set_title("UMAP colored by true label")
axes[0].set_xlabel("UMAP 1")
axes[0].set_ylabel("UMAP 2")

# Plot 2 - colored by kmeans cluster
scatter2 = axes[1].scatter(embedding_2d[:, 0], embedding_2d[:, 1],
                            c=cluster_ids, cmap="tab20", s=1, alpha=0.5)
axes[1].set_title("UMAP colored by KMeans cluster")
axes[1].set_xlabel("UMAP 1")
axes[1].set_ylabel("UMAP 2")

plt.tight_layout()
plt.savefig("umap_visualization.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved umap_visualization.png")

In [8]:
import plotly.express as px

embedding_2d = np.load("embedding_2d.npy")
# Load data
embeddings = np.load("embeddings/embeddings.npy")
labels = np.load("embeddings/labels.npy")
cluster_ids = np.load("cluster_ids.npy")
id_to_label = json.load(open("data/id_to_label.json"))

# Build dataframe
df = pd.DataFrame({
    "umap1":        embedding_2d[:, 0],
    "umap2":        embedding_2d[:, 1],
    "true_label": [id_to_label[str(int(i))] for i in labels],
    "cluster_id":   cluster_ids.astype(str),
})

# Plot colored by true label
fig1 = px.scatter(df, x="umap1", y="umap2",
                  color="true_label",
                  hover_data=["true_label", "cluster_id"],
                  title="UMAP colored by true label",
                  width=1000, height=800)
fig1.update_traces(marker=dict(size=3, opacity=0.6))


# Plot colored by kmeans cluster
fig2 = px.scatter(df, x="umap1", y="umap2",
                  color="cluster_id",
                  hover_data=["true_label", "cluster_id"],
                  title="UMAP colored by KMeans cluster",
                  width=1000, height=800)
fig2.update_traces(marker=dict(size=3, opacity=0.6))
fig1.write_html("umap_true_labels.html")
fig2.write_html("umap_kmeans.html")

In [3]:
label_names = json.load(open("data/labels.json"))

In [4]:
print(list(label_names.items())[:5])

[('apple_pie', 0), ('baby_back_ribs', 1), ('baklava', 2), ('beef_carpaccio', 3), ('beef_tartare', 4)]


In [5]:
id_to_label = {v: k for k, v in label_names.items()}

with open("data/id_to_label.json", "w") as f:
    json.dump(id_to_label, f, indent=2)
print("Saved id_to_label.json")

Saved id_to_label.json


In [19]:
from sklearn.metrics import silhouette_score
embeddings = np.load("embeddings/embeddings.npy")
id_to_label = json.load(open("data/id_to_label.json"))
true_labels = [id_to_label[str(int(i))] for i in labels]
cluster_ids = np.load("cluster_ids.npy")
score = silhouette_score(embeddings, cluster_ids, sample_size=5000)

In [20]:
score

0.2866295874118805

In [15]:
true_labels[:10]

['beignets',
 'beignets',
 'beignets',
 'beignets',
 'beignets',
 'beignets',
 'beignets',
 'beignets',
 'beignets',
 'beignets']

In [17]:
from sklearn.metrics.cluster import contingency_matrix
import numpy as np

matrix = contingency_matrix(true_labels, cluster_ids)
purity = np.sum(np.amax(matrix, axis=0)) / np.sum(matrix)

In [18]:
purity

0.8366732673267326

In [ ]:
df = pd.read_parquet('./data/test-00000-of-00001.parquet')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df_train = pd.read_parquet('./data/train-00000-of-00001.parquet')

In [ ]:
df_train.shape

In [ ]:
from datasets import load_dataset

ds = load_dataset("food101")

In [ ]:
df = ds["train"].to_pandas().head(10)

In [ ]:
df

In [ ]:
label_mapping = {
  "apple_pie": 0,
  "baby_back_ribs": 1,
  "baklava": 2,
  "beef_carpaccio": 3,
  "beef_tartare": 4,
  "beet_salad": 5,
  "beignets": 6,
  "bibimbap": 7,
  "bread_pudding": 8,
  "breakfast_burrito": 9,
  "bruschetta": 10,
  "caesar_salad": 11,
  "cannoli": 12,
  "caprese_salad": 13,
  "carrot_cake": 14,
  "ceviche": 15,
  "cheesecake": 16,
  "cheese_plate": 17,
  "chicken_curry": 18,
  "chicken_quesadilla": 19,
  "chicken_wings": 20,
  "chocolate_cake": 21,
  "chocolate_mousse": 22,
  "churros": 23,
  "clam_chowder": 24,
  "club_sandwich": 25,
  "crab_cakes": 26,
  "creme_brulee": 27,
  "croque_madame": 28,
  "cup_cakes": 29,
  "deviled_eggs": 30,
  "donuts": 31,
  "dumplings": 32,
  "edamame": 33,
  "eggs_benedict": 34,
  "escargots": 35,
  "falafel": 36,
  "filet_mignon": 37,
  "fish_and_chips": 38,
  "foie_gras": 39,
  "french_fries": 40,
  "french_onion_soup": 41,
  "french_toast": 42,
  "fried_calamari": 43,
  "fried_rice": 44,
  "frozen_yogurt": 45,
  "garlic_bread": 46,
  "gnocchi": 47,
  "greek_salad": 48,
  "grilled_cheese_sandwich": 49,
  "grilled_salmon": 50,
  "guacamole": 51,
  "gyoza": 52,
  "hamburger": 53,
  "hot_and_sour_soup": 54,
  "hot_dog": 55,
  "huevos_rancheros": 56,
  "hummus": 57,
  "ice_cream": 58,
  "lasagna": 59,
  "lobster_bisque": 60,
  "lobster_roll_sandwich": 61,
  "macaroni_and_cheese": 62,
  "macarons": 63,
  "miso_soup": 64,
  "mussels": 65,
  "nachos": 66,
  "omelette": 67,
  "onion_rings": 68,
  "oysters": 69,
  "pad_thai": 70,
  "paella": 71,
  "pancakes": 72,
  "panna_cotta": 73,
  "peking_duck": 74,
  "pho": 75,
  "pizza": 76,
  "pork_chop": 77,
  "poutine": 78,
  "prime_rib": 79,
  "pulled_pork_sandwich": 80,
  "ramen": 81,
  "ravioli": 82,
  "red_velvet_cake": 83,
  "risotto": 84,
  "samosa": 85,
  "sashimi": 86,
  "scallops": 87,
  "seaweed_salad": 88,
  "shrimp_and_grits": 89,
  "spaghetti_bolognese": 90,
  "spaghetti_carbonara": 91,
  "spring_rolls": 92,
  "steak": 93,
  "strawberry_shortcake": 94,
  "sushi": 95,
  "tacos": 96,
  "takoyaki": 97,
  "tiramisu": 98,
  "tuna_tartare": 99,
  "waffles": 100
}

In [ ]:
labels = {v: k for k, v in label_mapping.items()}

In [ ]:
labels

In [ ]:
import json
with open("./data/labels.json", "w") as f:
    json.dump(label_mapping, f)

In [ ]:
with open("./data/labels.json", "r") as f:
    data = json.load(f)

In [ ]:
item = 1000

In [ ]:
labels[ds['train'][item]['label']]

In [ ]:
ds['train'][item]['image'].size